In [13]:
## IMPORTS 
import evotoon
from data_classes import CatParam, IntParam, FloatParam


import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
#import seaborn as sns#

import random

In [14]:
## MAKE SEED
SEED = evotoon.make_seed(283)
separator = "--------------------------------------------------------"

In [15]:
## PROTOTYPE EXAMPLE WITH AntKnapsackClean-Master

# Env configuration
poblation_size = 10

# Parameter settings
float_params = [
	FloatParam("tau_max", 0.02, 4.0),
	FloatParam("rho", 0.001, 1.0),
	FloatParam("alpha", 1.0, 8.0),
	FloatParam("beta", 1.0, 8.0),	
]
int_params = [
	IntParam("total_ants", 2, 25),
]

initial_batch = evotoon.initialization(poblation_size, float_params, int_params)

# SET ENVIRONMENT FOR THE ILSMKP ALGORITHM TO TUNE
instance_list = [
	"./AntKnapsackClean-master/instances/weing1.txt",
	"./AntKnapsackClean-master/instances/weing2.txt",
	"./AntKnapsackClean-master/instances/weing3.txt",
	"./AntKnapsackClean-master/instances/weing4.txt",
	"./AntKnapsackClean-master/instances/weing7.txt",
]

seed_list = [SEED + i for i in range(len(instance_list))]
print(separator, "Instances running and seeds", separator)
for ins,seed in zip(instance_list,seed_list):
	print("instance:", ins, "seed:", seed)

function_kwargs = {
	"executable_path": "./AntKnapsackClean-master/AntKnapsack",
	"instance_list": instance_list,
	"seed_list": seed_list,
	"evaluations": 25,
	"tau_min": 0.01
}

print(pd.DataFrame(initial_batch))

-------------------------------------------------------- Instances running and seeds --------------------------------------------------------
instance: ./AntKnapsackClean-master/instances/weing1.txt seed: 283
instance: ./AntKnapsackClean-master/instances/weing2.txt seed: 284
instance: ./AntKnapsackClean-master/instances/weing3.txt seed: 285
instance: ./AntKnapsackClean-master/instances/weing4.txt seed: 286
instance: ./AntKnapsackClean-master/instances/weing7.txt seed: 287
    tau_max       rho     alpha      beta  total_ants
0  1.071048  0.371641  6.874519  6.688981          17
1  3.838395  0.982210  3.343147  1.353605          24
2  1.282402  0.069259  4.465537  6.081290           6
3  1.928077  0.612458  4.790863  2.781058           5
4  2.380992  0.221706  1.164632  5.522668          20
5  3.509404  0.721268  5.290972  5.092809           2
6  2.475350  0.566563  2.638268  3.118178          14
7  0.547996  0.856024  2.092802  7.740978          11
8  3.073571  0.109006  6.406744  1.81

In [26]:
# EVALUATE BATCH
# Make inputs / outputs
X = pd.DataFrame(initial_batch).values
y = evotoon.evaluate_batch(initial_batch, evotoon.execute_AntKnapsack, **function_kwargs)


# Create model
model = evotoon.create_model(X)
history = model.fit(X, y, epochs=50, verbose=0, validation_split = 0.2)

for i in range(10):
	# Generate configurations from model
	generated_X = evotoon.generate_configurations(X, float_params)

	# Evaluate current batch
	A = model.predict(generated_X)

	batch = [{a : b for a,b in zip(initial_batch[0].keys(), conf)} for conf in X]
	B = evotoon.evaluate_batch(batch, evotoon.execute_AntKnapsack, **function_kwargs)
	B = B.reshape(B.size, 1)

	# Select configurations for next step
	partition_A = np.argpartition(A[:, -1], -poblation_size//2)[-poblation_size//2:]
	X_generated = generated_X[partition_A]
	y_generated = A[partition_A]

	partition_B = np.argpartition(B[:, -1], - (poblation_size - poblation_size//2))[-(poblation_size-poblation_size//2):] 
	X_new = X[partition_B]
	y_new = B[partition_B]

	X = np.concatenate((X_generated, X_new), axis=0)
	y = np.concatenate((y_generated, y_new), axis=0)

	# Update model
	history = model.fit(X_new, y_new, epochs=50, verbose=0, validation_split = 0.2)

	print(f"MAX (percentual difference with optimal) found in iteration {i}:", np.amax(B))



MAX (percentual difference with optimal) found in iteration 0: -0.12039059999999999
MAX (percentual difference with optimal) found in iteration 1: -0.10310074
MAX (percentual difference with optimal) found in iteration 2: -0.071932618
MAX (percentual difference with optimal) found in iteration 3: -0.071932618
MAX (percentual difference with optimal) found in iteration 4: -0.071932618
MAX (percentual difference with optimal) found in iteration 5: -0.071932618
MAX (percentual difference with optimal) found in iteration 6: -0.071932618
MAX (percentual difference with optimal) found in iteration 7: -0.071932618
MAX (percentual difference with optimal) found in iteration 8: -0.071932618
MAX (percentual difference with optimal) found in iteration 9: -0.071932618


In [27]:
# Show best configurations found
print("Best configurations found:")
final_partition = np.argpartition(y_new[:, -1], - (poblation_size - poblation_size//2))[-(poblation_size-poblation_size//2):]
final_x = pd.DataFrame([{a : b for a,b in zip(initial_batch[0].keys(), conf)} for conf in X_new[final_partition]])
final_y = y_new[final_partition]
final_x["OPTIMAL DIFF"] = final_y * -1
final_x

Best configurations found:


,tau_max,rho,alpha,beta,total_ants,OPTIMAL DIFF
0,1.546187,0.232216,1.026973,4.387671,6.0,0.089613
1,0.733721,0.434102,3.749825,5.477929,14.0,0.083161
2,0.733540,0.362301,2.031375,3.618364,6.0,0.081007
3,0.733721,0.105051,3.045625,5.477929,14.0,0.080422
4,1.546187,0.069259,4.260664,3.618364,6.0,0.071933
